# 05 · Same Desk, Four Ways

**Module 06 · Agent Frameworks**

We built the same support desk four times: by hand, and in the OpenAI Agents SDK, LangGraph and CrewAI. They all use the same shop, the same model (`gpt-4o-mini`) and the same instructions, so any difference comes from the framework.

Each desk has the same function, `handle_email(email)`, so one test can run all of them. Like the evals in Module 05, the test checks **outcomes in the shop's data** (was a refund really created?) and not only the wording.

In [1]:
import asyncio

import compare
import shop_tools as shop

print("Ready")

Ready


## The inbox
Five emails, and what should happen to each.

In [2]:
INBOX = [
    {
        "name": "Order status",
        "email": "Hi, can you tell me where my order A1002 is? Thanks!",
        "specialist": "orders", "refunds": [], "text": "transit",
    },
    {
        "name": "Shipping policy",
        "email": "How much does express shipping cost?",
        "specialist": "policy", "refunds": [], "text": "12",
    },
    {
        "name": "Refund inside 30 days",
        "email": "The left earbud on my headphones (order A1001) stopped working. I would like my money back.",
        "specialist": "refunds", "refunds": ["A1001"], "text": "79",
    },
    {
        "name": "Refund after 30 days",
        "email": "I bought a coffee grinder (order A1003) three months ago and it broke. Please refund it.",
        "specialist": "refunds", "refunds": [], "text": "",
    },
    {
        "name": "Unknown order",
        "email": "Please refund my order A9999, I never received it.",
        "specialist": "refunds", "refunds": [], "text": "",
    },
]

In [3]:
print("The inbox has", len(compare.INBOX), "emails")
for case in compare.INBOX:
    print("-", case["name"])

The inbox has 5 emails
- Order status
- Shipping policy
- Refund inside 30 days
- Refund after 30 days
- Unknown order


## The check
For each email we compare the specialist that handled it, the refunds in the shop's data, and (where it matters) a word that must appear in the reply.

In [4]:
def check_result(case, result):
    """Compare what happened with what we expected. Returns a list of problems (empty = passed)."""
    problems = []

    if result["specialist"] != case["specialist"]:
        problems.append("went to '" + result["specialist"] + "', expected '" + case["specialist"] + "'")

    refunded = []
    for refund in result["refunds"]:
        refunded.append(refund["order_id"])
    if sorted(refunded) != sorted(case["refunds"]):
        problems.append("refunded " + str(refunded) + ", expected " + str(case["refunds"]))

    if case["text"] != "":
        if case["text"].lower() not in result["reply"].lower():
            problems.append("the reply does not contain '" + case["text"] + "'")

    return problems

## Run every desk
This makes about 40 small model calls and takes a couple of minutes.

> The desks use normal (blocking) calls, and Jupyter already runs an event loop, which the OpenAI Agents SDK and CrewAI do not like. So we run each desk in a worker thread with `asyncio.to_thread`. In a script (like `run_desk.py`) you simply call the function.

In [5]:
summaries = []

for desk_name in ["plain", "sdk", "langgraph", "crewai"]:
    print("=== " + desk_name + " ===")
    results = await asyncio.to_thread(compare.run_inbox, desk_name, show=True)
    summaries.append(compare.summarise(desk_name, results))
    print()

=== plain ===


ok     Order status           -> orders    437 tokens   4.7 s


ok     Shipping policy        -> policy    554 tokens   3.2 s


ok     Refund inside 30 days  -> refunds   893 tokens   3.1 s


ok     Refund after 30 days   -> refunds   590 tokens   2.1 s


ok     Unknown order          -> refunds   553 tokens   2.2 s

=== sdk ===


... (11 outputs left out)

ok     Shipping policy        -> policy    933 tokens   6.1 s


ok     Refund inside 30 days  -> refunds  1299 tokens   8.1 s


ok     Refund after 30 days   -> refunds   940 tokens   5.3 s


ok     Unknown order          -> refunds   901 tokens   5.4 s



## The scorecard

In [6]:
print("Desk".ljust(11) + "Correct".ljust(10) + "Avg tokens".ljust(12) + "Avg seconds".ljust(13) + "Lines of code")
print("-" * 56)
for item in summaries:
    print(item["desk"].ljust(11)
          + (str(item["passed"]) + "/" + str(item["total"])).ljust(10)
          + str(item["avg_tokens"]).ljust(12)
          + format(item["avg_seconds"], ".1f").ljust(13)
          + str(item["code_lines"]))

Desk       Correct   Avg tokens  Avg seconds  Lines of code
--------------------------------------------------------
plain      5/5       605         3.1          81
sdk        5/5       736         3.5          46
langgraph  5/5       581         2.5          62
crewai     5/5       977         6.1          69


**How to read it** (your numbers will differ a little, because models are not deterministic):

- **Correct.** All four should get the same answers on this easy desk. A framework does not make a model smarter. It changes how you *build* the system.
- **Tokens.** Compare each desk with plain Python. A framework that writes longer hidden prompts or adds extra steps (such as a triage handoff) uses more tokens, and more tokens means a higher bill at scale. A framework that adds nothing extra uses about the same.
- **Seconds.** Each extra model call adds waiting time.
- **Lines of code.** This counts the whole desk file, and the hand-built file includes its own tool descriptions, loop and router. Fewer lines is not always better: the hidden code is still there, inside the framework.

## Which one should you pick?

| If you need... | Look at | Because |
|---|---|---|
| The quickest start, on OpenAI models | **OpenAI Agents SDK** | Few ideas, handoffs and tracing are built in |
| Control, branches, loops, saved progress, human approval | **LangGraph** | You draw the flow, and the state can be paused and resumed |
| A pipeline of roles (research, write, review) that reads like a job description | **CrewAI** | Agents, tasks and crews map to how people describe teams |
| One agent, full visibility, lowest cost | **Plain Python** | No framework to learn, nothing hidden |
| No code at all | A low-code tool (n8n, Flowise, Dify) | Visual canvas, for people who do not program |

A few honest rules:
1. **Start without a framework.** Add one when you feel the pain it solves.
2. **Pick by the shape of your problem**, not by popularity.
3. **Keep your business logic outside the framework.** Notice that all four desks share the same `shop_tools.py`. If the tools are plain functions, switching frameworks only means rewriting the thin wrapper.
4. **Pin versions and run your evals** after every upgrade.

## What none of them gave us
Look back at Module 05. The desks here have **no input guards, no refund limit with human approval and no output checks**. A framework is not a safety system. The guardrails we built are *plain checks around the desk*, so they can wrap **any** of these frameworks.

That is what the capstone does next: put the Module 05 guardrails around the support desk in each framework, run the same eval suite on all of them, and compare real numbers.